# LangChain Fundamentals: Part 4
## Advanced Patterns: Summarization & Structured Output

### Learning Objectives
By the end of this notebook, you'll understand:
1. **Batch Processing** - Processing multiple items efficiently
2. **Summarization** - Condensing large texts into summaries
3. **Structured Output** - Using Pydantic models for predictable output
4. **Tool Binding** - Using LLM tools for structured extraction
5. **Real-world Patterns** - Building practical applications

## Problem: Unstructured Data

### The Challenge:
LLMs return text. But we often need **structured data** for systems.

### Without Structure:
```python
response = llm.invoke("Extract: name, email, phone from: ...")
# Returns: "The name is John Smith, his email is john@example.com..."
# ❌ Hard to parse programmatically
```

### With Structure (Pydantic):
```python
class Contact(BaseModel):
    name: str
    email: str
    phone: str

result = llm.invoke(...)  # Returns Contact object
# ✅ result.name, result.email, result.phone ready to use
```

## Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore', category=PendingDeprecationWarning)

from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

from langchain_openai import ChatOpenAI, OpenAIEmbeddings

llm = ChatOpenAI(
    model="gpt-4o",
    temperature=0
)

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

print("✓ Models initialized!")

## Step 1: Loading Data

Let's load some news feed data to work with:

In [ ]:
import pandas as pd

# Load news feed data from Excel
feeds = pd.read_excel('feeds.xlsx')

print(f"Loaded {len(feeds)} news articles")
print(f"\nColumns: {list(feeds.columns)}")
print(f"\nFirst article:")
print(f"Title: {feeds.iloc[0]['title']}")
print(f"Description: {feeds.iloc[0]['description'][:100]}...")

## Step 2: Preparing Data for Batch Processing

Batch processing allows us to process multiple items efficiently and in parallel.

In [ ]:
# Combine title and description
feeds['full_text'] = feeds['title'] + " " + feeds['description']

# Create batch inputs - a list of dictionaries
# Each dictionary has the inputs our chain expects
batch_inputs = [
    {'full_text': text} 
    for text in feeds['full_text'][:20]  # First 20 articles
]

print(f"Created {len(batch_inputs)} batch inputs")
print(f"\nExample input:")
print(batch_inputs[0])

## Pattern 1: Batch Summarization

### Use Case: Summarizing news articles

Goal: Reduce long articles to key points (50 words each)

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Create a summarization prompt
summary_template = """You are a professional summarizer. 
Summarize the following text in exactly 50 words or fewer:

Text: {full_text}

Summary:"""

summary_prompt = PromptTemplate(
    template=summary_template,
    input_variables=['full_text']
)

# Create the chain
summary_chain = summary_prompt | llm | StrOutputParser()

# Process the first 5 articles
print("Batch Summarization Results:\n")
print("="*80)

summaries = summary_chain.batch(batch_inputs[:5])

for i, (input_data, summary) in enumerate(zip(batch_inputs[:5], summaries), 1):
    print(f"\n{i}. Original ({len(input_data['full_text'])} chars):")
    print(f"   {input_data['full_text'][:100]}...")
    print(f"\n   Summary ({len(summary)} chars):")
    print(f"   {summary}")
    print("-"*80)

## Pattern 2: Batch Processing at Scale

The power of batch() is processing many items efficiently:

In [ ]:
# Process all 20 articles
print("Processing 20 articles in batch...")
all_summaries = summary_chain.batch(batch_inputs)

print(f"✓ Processed {len(all_summaries)} articles")
print(f"\nFirst 3 summaries:")

for i, summary in enumerate(all_summaries[:3], 1):
    print(f"\n{i}. {summary}")

## Pattern 3: Structured Output with Pydantic

### What is Pydantic?

Pydantic is a Python library for **data validation** using type hints.

```python
from pydantic import BaseModel, Field

class Person(BaseModel):
    name: str
    age: int
    email: str
```

### Why use it?
- ✅ Validates data types
- ✅ Provides schema for LLM
- ✅ Generates structured JSON
- ✅ Type-safe output

In [ ]:
from pydantic import BaseModel, Field

# Define the structure of our output
class NewsAnalysis(BaseModel):
    """Structured analysis of a news article"""
    title: str = Field(
        ..., 
        description="Article title (max 100 chars)"
    )
    summary: str = Field(
        ..., 
        description="Summary in 50 words or less"
    )
    sentiment: str = Field(
        ..., 
        description="Sentiment: positive, negative, or neutral"
    )
    key_topics: list = Field(
        ..., 
        description="List of 3-5 key topics mentioned"
    )
    relevance_score: int = Field(
        ..., 
        description="Relevance score from 1-10"
    )

print("✓ NewsAnalysis model defined")
print(f"\nModel schema:")
print(NewsAnalysis.model_json_schema())

## Pattern 4: Tool Binding for Structured Output

We can bind Pydantic models to the LLM as "tools" to get structured output:

In [ ]:
from langchain_core.output_parsers import PydanticToolsParser

# Create a prompt for structured extraction
extraction_template = """Extract structured information from the following news article.
Analyze the article and provide all required fields.

Article: {full_text}"""

extraction_prompt = PromptTemplate.from_template(extraction_template)

# Bind the model as a tool to the LLM
llm_with_tool = llm.bind_tools([NewsAnalysis])

# Create the parser
parser = PydanticToolsParser(tools=[NewsAnalysis])

# Create the chain
extraction_chain = extraction_prompt | llm_with_tool | parser

print("✓ Structured extraction chain created")

## Pattern 5: Testing Structured Output

Let's test the structured extraction on a few articles:

In [ ]:
# Process the first 3 articles
print("Structured Analysis Results:\n")
print("="*80)

structured_results = extraction_chain.batch(batch_inputs[:3])

for i, result in enumerate(structured_results, 1):
    # result is a list of extracted models
    if result:
        analysis = result[0]  # Get the first (and usually only) result
        
        print(f"\n{i}. Article Analysis:")
        print(f"   Title: {analysis.title}")
        print(f"   Summary: {analysis.summary}")
        print(f"   Sentiment: {analysis.sentiment}")
        print(f"   Topics: {', '.join(analysis.key_topics)}")
        print(f"   Relevance: {analysis.relevance_score}/10")
        print("-"*80)

## Pattern 6: Domain-Specific Structured Output

Here's a real-world example: Trading recommendations

In [ ]:
class TradingRecommendation(BaseModel):
    """Structured trading recommendation"""
    asset_name: str = Field(
        ..., 
        description="Name of the asset to invest in"
    )
    current_situation: str = Field(
        ..., 
        description="Current market situation for this asset"
    )
    macro_environment: str = Field(
        ..., 
        description="Description of macroeconomic environment"
    )
    geopolitical_risks: str = Field(
        ..., 
        description="Any geopolitical considerations"
    )
    market_sentiment: str = Field(
        ..., 
        description="Current market sentiment toward this asset"
    )
    risk_level: str = Field(
        ..., 
        description="Risk level: low, medium, or high"
    )
    recommendation: str = Field(
        ..., 
        description="Trading recommendation: buy, hold, or sell"
    )

print("✓ TradingRecommendation model defined")

## Pattern 7: Trading Recommendation Chain

In [ ]:
# Create a trading recommendation prompt
trading_template = """You are an expert trader and financial analyst.

Based on the following news article, provide a structured trading recommendation.
Consider the asset class mentioned, market conditions, and risks.

Article: {full_text}"""

trading_prompt = PromptTemplate.from_template(trading_template)

# Bind the trading model as a tool
llm_with_trading_tool = llm.bind_tools([TradingRecommendation])

# Create the parser
trading_parser = PydanticToolsParser(tools=[TradingRecommendation])

# Create the chain
trading_chain = trading_prompt | llm_with_trading_tool | trading_parser

print("✓ Trading recommendation chain created")

## Pattern 8: Batch Trading Recommendations

In [ ]:
# Get trading recommendations for the first 3 articles
print("Trading Recommendations:\n")
print("="*80)

trading_results = trading_chain.batch(batch_inputs[:3])

for i, result in enumerate(trading_results, 1):
    if result:
        rec = result[0]
        
        print(f"\n{i}. Trading Recommendation")
        print(f"   Asset: {rec.asset_name}")
        print(f"   Situation: {rec.current_situation[:100]}...")
        print(f"   Macro Environment: {rec.macro_environment[:100]}...")
        print(f"   Geopolitical Risks: {rec.geopolitical_risks[:100]}...")
        print(f"   Sentiment: {rec.market_sentiment}")
        print(f"   Risk Level: {rec.risk_level}")
        print(f"   ➜ Recommendation: {rec.recommendation}")
        print("-"*80)

## Pattern 9: Combining Patterns - Full Pipeline

Let's create a comprehensive pipeline that:
1. Summarizes articles
2. Analyzes them
3. Provides trading recommendations

In [ ]:
def comprehensive_analysis(articles_list, num_articles=5):
    """
    Comprehensive analysis pipeline:
    1. Summarize each article
    2. Analyze sentiment and topics
    3. Generate trading recommendations
    """
    
    # Prepare inputs
    inputs = [
        {'full_text': text} 
        for text in articles_list[:num_articles]
    ]
    
    print(f"Processing {len(inputs)} articles...\n")
    
    # Step 1: Summarize
    summaries = summary_chain.batch(inputs)
    
    # Step 2: Analyze
    analyses = extraction_chain.batch(inputs)
    
    # Step 3: Get trading recommendations
    recommendations = trading_chain.batch(inputs)
    
    # Combine results
    results = []
    for i, (inp, summary, analysis, recommendation) in enumerate(
        zip(inputs, summaries, analyses, recommendations), 1
    ):
        result = {
            'article_index': i,
            'original_text': inp['full_text'][:100],
            'summary': summary,
            'analysis': analysis[0] if analysis else None,
            'recommendation': recommendation[0] if recommendation else None
        }
        results.append(result)
    
    return results

print("✓ Comprehensive analysis pipeline defined")

## Pattern 10: Running the Complete Pipeline

In [ ]:
# Run the comprehensive analysis
results = comprehensive_analysis(batch_inputs, num_articles=3)

print("\n" + "="*80)
print("COMPREHENSIVE ANALYSIS RESULTS")
print("="*80)

for result in results:
    print(f"\n📰 Article {result['article_index']}")
    print(f"\n   Original: {result['original_text']}...")
    print(f"\n   📝 Summary: {result['summary']}")
    
    if result['analysis']:
        analysis = result['analysis']
        print(f"\n   📊 Analysis:")
        print(f"      • Sentiment: {analysis.sentiment}")
        print(f"      • Topics: {', '.join(analysis.key_topics)}")
        print(f"      • Relevance: {analysis.relevance_score}/10")
    
    if result['recommendation']:
        rec = result['recommendation']
        print(f"\n   💰 Trading:")
        print(f"      • Asset: {rec.asset_name}")
        print(f"      • Risk: {rec.risk_level}")
        print(f"      • Action: {rec.recommendation}")
    
    print("\n" + "-"*80)

## Key Takeaways

### Batch Processing:
- Process multiple items in parallel
- More efficient than sequential processing
- Returns list of results in same order

### Structured Output:
- Use Pydantic models to define schema
- Bind models to LLM as "tools"
- Get validated, type-safe output

### Real-world Patterns:
- Summarization for content processing
- Structured extraction for data organization
- Domain-specific models for specialized tasks

### Pipeline Architecture:
- Chain multiple operations together
- Reuse components across applications
- Scale from prototypes to production

## Comparison: Text Output vs Structured Output

### Text Output (Unstructured):
```python
response = llm.invoke("Summarize this article")
# Returns: "The article discusses..." (string)
# ❌ Hard to parse programmatically
# ❌ Type-unsafe
```

### Structured Output (Pydantic):
```python
response = chain.invoke(...)
# Returns: NewsAnalysis object
# ✅ response.summary (guaranteed string)
# ✅ response.sentiment (one of: positive/negative/neutral)
# ✅ response.key_topics (guaranteed list)
# ✅ Type-safe and predictable
```

## Practice Exercises

### Exercise 1: Custom Pydantic Model
Create a new Pydantic model for a different domain (e.g., product reviews, customer feedback, research papers).

### Exercise 2: Larger Batch
Process all 20 articles and analyze the results. What patterns do you notice?

### Exercise 3: Chain Modification
Modify the prompts to get different analyses (e.g., focus on risks, opportunities, or compliance).

### Exercise 4: Error Handling
What happens if the LLM can't extract the required fields? Add error handling.

### Exercise 5: Performance Analysis
Time how long it takes to process 10 vs 20 articles. Is batch processing faster than sequential?